# 01 — Exploratory Data Analysis

Understand the DermNet dataset before training:
- Class distribution & imbalance
- Image sample grid per class
- Image size statistics
- Class name mapping

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from dermnet.config import load_config

cfg = load_config('../configs/default.yaml')
data_dir = Path('../') / cfg.data.data_dir
print(f'Data directory: {data_dir}')
print(f'Exists: {data_dir.exists()}')

## 1. Class Distribution

In [ ]:
class_counts = {}
for class_dir in sorted(data_dir.iterdir()):
    if class_dir.is_dir():
        imgs = list(class_dir.glob('*.[jJpP][pPnN][gG]'))
        class_counts[class_dir.name] = len(imgs)

counts_df = pd.Series(class_counts).sort_values(ascending=False)
print(f'Total images : {counts_df.sum():,}')
print(f'Total classes: {len(counts_df)}')
print(f'Min: {counts_df.min()} ({counts_df.idxmin()})')
print(f'Max: {counts_df.max()} ({counts_df.idxmax()})')
print(f'Imbalance ratio: {counts_df.max() / counts_df.min():.1f}x')

In [ ]:
Path('../outputs/figures').mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(12, 8))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(counts_df)))
counts_df.sort_values().plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel('Number of Images')
ax.set_title('Class Distribution — DermNet Dataset', fontsize=14)
ax.axvline(counts_df.mean(), color='red', linestyle='--',
           label=f'Mean ({counts_df.mean():.0f})')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('This imbalance motivates: WeightedRandomSampler + class-weighted loss')

## 2. Sample Images for Key Classes

In [ ]:
# Show 5 sample images for the 3 primary target classes
target_keywords = ['eczema', 'psoriasis', 'fungal']
n_samples = 5

matched_dirs = []
for kw in target_keywords:
    for d in sorted(data_dir.iterdir()):
        if d.is_dir() and kw.lower() in d.name.lower():
            matched_dirs.append(d)
            break

fig, axes = plt.subplots(len(matched_dirs), n_samples, figsize=(15, 3*len(matched_dirs)))
if len(matched_dirs) == 1:
    axes = [axes]

for row, class_dir in enumerate(matched_dirs):
    imgs = sorted(class_dir.glob('*.[jJpP][pPnN][gG]'))[:n_samples]
    for col, img_path in enumerate(imgs):
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        axes[row][col].imshow(img)
        axes[row][col].axis('off')
        if col == 0:
            short = class_dir.name[:35] + ('...' if len(class_dir.name) > 35 else '')
            axes[row][col].set_ylabel(short, fontsize=7)

plt.suptitle('Sample Images: 3 Target Disease Classes', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/figures/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Image Size Statistics

In [ ]:
import random

all_images = [p for d in data_dir.iterdir() if d.is_dir()
              for p in d.glob('*.[jJpP][pPnN][gG]')]
sample = random.sample(all_images, min(500, len(all_images)))
heights, widths = [], []
for p in sample:
    img = cv2.imread(str(p))
    if img is not None:
        heights.append(img.shape[0])
        widths.append(img.shape[1])

print(f'Height: mean={np.mean(heights):.0f}  std={np.std(heights):.0f}')
print(f'Width:  mean={np.mean(widths):.0f}  std={np.std(widths):.0f}')
print(f'Resizing all to {cfg.data.image_size}x{cfg.data.image_size}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(heights, bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(cfg.data.image_size, color='red', linestyle='--',
                label=f'Resize ({cfg.data.image_size}px)')
axes[0].set_title('Height Distribution'); axes[0].legend()
axes[1].hist(widths, bins=40, color='darkorange', edgecolor='white')
axes[1].axvline(cfg.data.image_size, color='red', linestyle='--')
axes[1].set_title('Width Distribution')
plt.tight_layout()
plt.savefig('../outputs/figures/image_size_dist.png', dpi=150)
plt.show()